In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio
from rasterio.transform import from_bounds
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import gc
import warnings
warnings.filterwarnings('ignore')

from src.data_utils import load_config, ensure_dir

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090


In [2]:
cfg           = load_config('../configs/config.yaml')
processed_dir = Path('../data/processed')
model_dir     = Path('../models')
results_dir   = ensure_dir('../results')
fig_dir       = ensure_dir('../results/figures')

# 加载训练好的 resnet34 U-Net 模型
class UNetModel:
    pass

unet = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=7,
    classes=1,
    activation='sigmoid'
).to(device)

unet.load_state_dict(torch.load(model_dir / 'unet_cfactor_2024_04.pth', map_location=device))
unet.eval()

total_params = sum(p.numel() for p in unet.parameters())
print(f'模型加载成功')
print(f'参数量: {total_params:,}')
print(f'编码器: resnet34')
print(f'训练精度: R²=0.82')

# NSW 全州范围
bbox_nsw = cfg['region']['bbox']
print(f'\nNSW 全州范围:')
print(f'  经度: {bbox_nsw["lon_min"]}°E → {bbox_nsw["lon_max"]}°E')
print(f'  纬度: {bbox_nsw["lat_min"]}°S → {bbox_nsw["lat_max"]}°S')

模型加载成功
参数量: 24,448,913
编码器: resnet34
训练精度: R²=0.82

NSW 全州范围:
  经度: 140.999°E → 153.639°E
  纬度: -37.505°S → -28.157°S


In [3]:
from pystac_client import Client
import planetary_computer as pc
import stackstac
from tqdm import tqdm

STAC_URL   = cfg['sentinel2']['stac_endpoint']
EPSG       = 32755
CLOUD_MAX  = 5
TIME_RANGE = '2024-04-01/2024-05-01'

# NSW 全州 BBOX
BBOX_NSW = [
    bbox_nsw['lon_min'], bbox_nsw['lat_min'],
    bbox_nsw['lon_max'], bbox_nsw['lat_max']
]

print('搜索 NSW 全州 Sentinel-2 场景...')
catalog   = Client.open(STAC_URL)
search    = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX_NSW,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': CLOUD_MAX}}
)
raw_items = list(search.items())
print(f'找到场景数: {len(raw_items)}')

# 按日期分组
raw_dates = {}
for item in raw_items:
    date = item.datetime.strftime('%Y-%m-%d')
    if date not in raw_dates:
        raw_dates[date] = []
    raw_dates[date].append(item)

print(f'共 {len(raw_dates)} 个日期:')
for date, sitems in sorted(raw_dates.items()):
    print(f'  {date}: {len(sitems)} 景')

搜索 NSW 全州 Sentinel-2 场景...
找到场景数: 684
共 30 个日期:
  2024-04-01: 14 景
  2024-04-02: 23 景
  2024-04-03: 14 景
  2024-04-04: 18 景
  2024-04-05: 28 景
  2024-04-06: 2 景
  2024-04-07: 19 景
  2024-04-08: 8 景
  2024-04-09: 24 景
  2024-04-10: 20 景
  2024-04-11: 43 景
  2024-04-12: 26 景
  2024-04-13: 16 景
  2024-04-14: 55 景
  2024-04-15: 13 景
  2024-04-16: 39 景
  2024-04-17: 41 景
  2024-04-18: 1 景
  2024-04-19: 12 景
  2024-04-20: 25 景
  2024-04-21: 26 景
  2024-04-22: 51 景
  2024-04-23: 34 景
  2024-04-24: 20 景
  2024-04-25: 2 景
  2024-04-26: 37 景
  2024-04-27: 35 景
  2024-04-28: 13 景
  2024-04-30: 3 景
  2024-05-01: 22 景


In [4]:
# 选景数最多的5个日期，覆盖全州
best_dates = sorted(raw_dates.items(), key=lambda x: len(x[1]), reverse=True)[:5]

print('选取日期:')
for date, sitems in best_dates:
    print(f'  {date}: {len(sitems)} 景')

print(f'\n预计下载量: {sum(len(s) for _, s in best_dates)} 景')
print(f'预计下载时间: 2~4小时')
print(f'预计内存占用: 200~300GB')

选取日期:
  2024-04-14: 55 景
  2024-04-22: 51 景
  2024-04-11: 43 景
  2024-04-17: 41 景
  2024-04-16: 39 景

预计下载量: 229 景
预计下载时间: 2~4小时
预计内存占用: 200~300GB


In [5]:
from src.indices import calculate_all_indices

# NSW全州分块策略
# 经度方向分4块，纬度方向分3块，共12个子区域
LON_SPLITS = 4
LAT_SPLITS = 3

lon_min = bbox_nsw['lon_min']  # 140.999
lon_max = bbox_nsw['lon_max']  # 153.639
lat_min = bbox_nsw['lat_min']  # -37.505
lat_max = bbox_nsw['lat_max']  # -28.157

lon_step = (lon_max - lon_min) / LON_SPLITS
lat_step = (lat_max - lat_min) / LAT_SPLITS

# 生成所有子区域
tiles = []
for i in range(LON_SPLITS):
    for j in range(LAT_SPLITS):
        tile = {
            'lon_min': lon_min + i * lon_step,
            'lon_max': lon_min + (i+1) * lon_step,
            'lat_min': lat_min + j * lat_step,
            'lat_max': lat_min + (j+1) * lat_step,
            'tile_id': f'tile_{i}_{j}'
        }
        tiles.append(tile)

print(f'分块策略: {LON_SPLITS}×{LAT_SPLITS} = {len(tiles)} 个子区域')
print(f'每块大小: {lon_step:.3f}°×{lat_step:.3f}° ≈ {lon_step*111:.0f}km×{lat_step*111:.0f}km')
print(f'\n子区域列表:')
for t in tiles:
    print(f'  {t["tile_id"]}: [{t["lon_min"]:.3f},{t["lat_min"]:.3f}] → [{t["lon_max"]:.3f},{t["lat_max"]:.3f}]')

分块策略: 4×3 = 12 个子区域
每块大小: 3.160°×3.116° ≈ 351km×346km

子区域列表:
  tile_0_0: [140.999,-37.505] → [144.159,-34.389]
  tile_0_1: [140.999,-34.389] → [144.159,-31.273]
  tile_0_2: [140.999,-31.273] → [144.159,-28.157]
  tile_1_0: [144.159,-37.505] → [147.319,-34.389]
  tile_1_1: [144.159,-34.389] → [147.319,-31.273]
  tile_1_2: [144.159,-31.273] → [147.319,-28.157]
  tile_2_0: [147.319,-37.505] → [150.479,-34.389]
  tile_2_1: [147.319,-34.389] → [150.479,-31.273]
  tile_2_2: [147.319,-31.273] → [150.479,-28.157]
  tile_3_0: [150.479,-37.505] → [153.639,-34.389]
  tile_3_1: [150.479,-34.389] → [153.639,-31.273]
  tile_3_2: [150.479,-31.273] → [153.639,-28.157]


In [9]:
# 针对西部tile重新搜索，放宽云量到20%
print('为西部tile重新搜索场景...')
catalog_west = Client.open(STAC_URL)

BBOX_WEST = [140.999, -37.505, 144.159, -28.157]

search_west = catalog_west.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX_WEST,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': 20}}
)

raw_items_west = list(search_west.items())
print(f'西部区域找到场景数: {len(raw_items_west)}')

raw_dates_west = {}
for item in raw_items_west:
    date = item.datetime.strftime('%Y-%m-%d')
    if date not in raw_dates_west:
        raw_dates_west[date] = []
    raw_dates_west[date].append(item)

best_dates_west = sorted(
    raw_dates_west.items(),
    key=lambda x: len(x[1]), reverse=True
)[:5]

print(f'\n西部最优日期:')
for date, sitems in best_dates_west:
    print(f'  {date}: {len(sitems)} 景')

为西部tile重新搜索场景...
西部区域找到场景数: 367

西部最优日期:
  2024-04-22: 47 景
  2024-04-17: 38 景
  2024-04-27: 37 景
  2024-04-12: 30 景
  2024-04-20: 28 景


In [ ]:
import gc
from pystac_client import Client
import planetary_computer as pc

gc.collect()

# 重新搜索，保存原始未签名items
print('重新搜索东部NSW场景...')
catalog_fresh = Client.open(STAC_URL)
search_fresh  = catalog_fresh.search(
    collections=['sentinel-2-l2a'],
    bbox=[144.159, -37.505, 153.639, -28.157],
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': CLOUD_MAX}}
)
raw_items_fresh = list(search_fresh.items())  # 原始未签名
print(f'找到场景数: {len(raw_items_fresh)}')

raw_dates_fresh = {}
for item in raw_items_fresh:
    date = item.datetime.strftime('%Y-%m-%d')
    if date not in raw_dates_fresh:
        raw_dates_fresh[date] = []
    raw_dates_fresh[date].append(item)

# 选景数最多的5个日期（未签名）
best_dates_east = sorted(
    raw_dates_fresh.items(),
    key=lambda x: len(x[1]), reverse=True
)[:5]

print(f'\n选取日期:')
for date, sitems in best_dates_east:
    print(f'  {date}: {len(sitems)} 景')

# 修改process_tile确保每次循环内即时签名
def process_tile(tile, best_dates, epsg=32755):
    bbox_tile = [tile['lon_min'], tile['lat_min'],
                 tile['lon_max'], tile['lat_max']]
    tile_id   = tile['tile_id']

    out_path = output_dir / f'NSW_{tile_id}_features_2024_04.npy'
    if out_path.exists():
        print(f'  {tile_id} 已存在，跳过')
        return True

    ndvi_max_global = None
    best_day_bands  = None
    height = width  = None
    transform_ref   = None

    for date, raw_scene_items in best_dates:
        try:
            # 每次下载前即时签名
            scene_items = [pc.sign(item) for item in raw_scene_items]

            day_stack = stackstac.stack(
                scene_items,
                assets=['B02','B03','B04','B08','B11'],
                bounds_latlon=bbox_tile,
                resolution=10,
                dtype='float64',
                fill_value=0,
                rescale=False,
                epsg=epsg
            )
            day_data = day_stack.compute()

            if transform_ref is None:
                transform_ref = day_stack.transform
                height, width = day_data.shape[2], day_data.shape[3]
                ndvi_max_global = np.full((height, width), -999.0, dtype='float32')
                best_day_bands  = np.full((height, width, 5), np.nan, dtype='float32')

            b02 = np.nanmax(day_data.sel(band='B02').values / 10000.0, axis=0)
            b03 = np.nanmax(day_data.sel(band='B03').values / 10000.0, axis=0)
            b04 = np.nanmax(day_data.sel(band='B04').values / 10000.0, axis=0)
            b08 = np.nanmax(day_data.sel(band='B08').values / 10000.0, axis=0)
            b11 = np.nanmax(day_data.sel(band='B11').values / 10000.0, axis=0)

            del day_data, day_stack
            gc.collect()

            valid    = (b04 > 0) & (b08 > 0)
            ndvi_day = np.where(valid,
                                (b08 - b04) / (b08 + b04 + 1e-8),
                                -999.0).astype('float32')

            update_mask     = ndvi_day > ndvi_max_global
            ndvi_max_global = np.where(update_mask, ndvi_day, ndvi_max_global)

            update_mask_3d = update_mask[:, :, np.newaxis]
            new_bands = np.stack([
                np.where(valid, b02, np.nan),
                np.where(valid, b03, np.nan),
                np.where(valid, b04, np.nan),
                np.where(valid, b08, np.nan),
                np.where(valid, b11, np.nan),
            ], axis=-1).astype('float32')

            best_day_bands = np.where(update_mask_3d, new_bands, best_day_bands)

            del b02, b03, b04, b08, b11, ndvi_day, new_bands, valid, update_mask
            gc.collect()

            valid_pct = (ndvi_max_global > -999).mean() * 100
            print(f'    {date} 完成，累计有效像素: {valid_pct:.1f}%')

        except Exception as e:
            print(f'    {date} 跳过: {e}')

    if ndvi_max_global is None or (ndvi_max_global == -999).all():
        print(f'  {tile_id} 无有效数据')
        return False

    ndvi_max_global[ndvi_max_global == -999] = np.nan

    B02 = best_day_bands[:, :, 0]
    B03 = best_day_bands[:, :, 1]
    B04 = best_day_bands[:, :, 2]
    B08 = best_day_bands[:, :, 3]
    B11 = best_day_bands[:, :, 4]

    indices = calculate_all_indices(
        {'blue': B02, 'green': B03, 'red': B04, 'nir': B08, 'swir': B11}
    )

    feature_stack = np.stack([
        indices['NDVI'], indices['NDWI'],
        indices['BSI'],  indices['SAVI'],
        B04, B08, B11
    ], axis=-1).astype('float32')

    np.save(out_path, feature_stack)

    meta_path = output_dir / f'NSW_{tile_id}_meta.npy'
    np.save(meta_path, {
        'transform': transform_ref,
        'crs': f'EPSG:{epsg}',
        'height': height,
        'width': width,
        'bbox': bbox_tile
    }, allow_pickle=True)

    valid_pct = np.isfinite(feature_stack[:, :, 0]).mean() * 100
    print(f'  {tile_id} 完成 | 形状:{feature_stack.shape} | 有效像素:{valid_pct:.1f}%')

    del feature_stack, best_day_bands, ndvi_max_global, B02, B03, B04, B08, B11
    gc.collect()
    return True

# 逐tile处理
print('\n开始处理东部NSW 9个子区域...')
print('='*50)

for i, tile in enumerate(tiles_east):
    print(f'\n[{i+1}/{len(tiles_east)}] 处理 {tile["tile_id"]}...')
    process_tile(tile, best_dates_east)

print('\n东部NSW全部处理完成')

重新搜索东部NSW场景...
找到场景数: 493

选取日期:
  2024-04-14: 53 景
  2024-04-11: 43 景
  2024-04-16: 39 景
  2024-04-26: 37 景
  2024-04-23: 34 景

开始处理东部NSW 9个子区域...

[1/9] 处理 tile_1_0...
  tile_1_0 已存在，跳过

[2/9] 处理 tile_1_1...
    2024-04-14 跳过: Error reading Window(col_off=8192, row_off=34816, width=1024, height=89) from 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/55/H/CC/2024/04/14/S2A_MSIL2A_20240414T002101_N0510_R116_T55HCC_20240414T060800.SAFE/GRANULE/L2A_T55HCC_A046016_20240414T002150/IMG_DATA/R10m/T55HCC_20240414T002101_B02_10m.tif?st=2026-05-19T05%3A37%3A37Z&se=2026-05-20T06%3A22%3A37Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-05-20T05%3A27%3A42Z&ske=2026-05-27T05%3A27%3A42Z&sks=b&skv=2025-07-05&sig=cP/IIpNLHlGaNCoHe4AGoT3E1vCKgbp2k97/ApDchNs%3D': RasterioIOError('Read failed. See previous exception for details.')
    2024-04-11 完成，累计有效像素: 30.5%
    2024-04-16 完成，累计有效像素: 30.6%
    2024-04-26 跳过: Error r